# ANIMATED PLOTS WITH `matplotlib`

We will animate the plot of the projectile trajectory.  Animating plots requires loading the `animation` module from `matplotlib`.

The main difference is that instead of a simple `plt.plot(...)` command, we need to interact with the elements forming a plot. To understand the internals of the plot object take a look at this [matplotlib FAQ](https://matplotlib.org/faq/usage_faq.html).

To produce the data to plot, we use a simple version of the projectile example with all values fixed.  Then we focus on the plotting part in the examples that follow.

In [ ]:
#%matplotlib inline does not behave well with animate on my machine
#%matplotlib notebook does, but it does not work smoothly with colab...
# To make things work on colab stick to inline, and take the additional steps commented out after plt.show()
%matplotlib notebook

# The ordering of these import can affect the behaviour of the notebook when
# using older Python interpreters.  Place the numpy import first for safety.
from numpy import arange
import math as m
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# All the rest here should be familiar by now...
g = 9.8 # m/s^2
h = 10. # m
theta = m.radians(30.)
v0 = 30. # m/s
x0 = 0   # m
y0 = h   # m
dt = 0.1 # s
max_t = 1000 # s

v0x = v0*m.cos(theta)
v0y = v0*m.sin(theta)
print(f"v0_x: {v0x:.1f} m/s \t v0_y: {v0y:.1f} m/s")

# Evolution function: returns 3 values: t, x(t), y(t)
def pos(t):
    return t, x0 + v0x*t, y0 + v0y*t - 0.5*g*t*t

# Use a list of 3-element tuples to perform a single comprehension
# Stop when y(t)=pos(t)[2] < 0
trajectory = [pos(t) for t in arange(0., max_t, dt) if pos(t)[2]>=0.]
t = [el[0] for el in trajectory]
x = [el[1] for el in trajectory]
y = [el[2] for el in trajectory]

# Highest point of the trajectory
print(f"Max height: {max(y):.2f} m at x = {x[list(y).index(max(y))]:.2f} m")

Here is a static version of the plot we want to animate.

We use the `subplot` function. In principle a figure can now contain multiple plots and
`ax.plot()` returns a list of objects, even if it contains only one object ([matplotlib.axes.Axes.plot](https://matplotlib.org/3.3.3/api/_as_gen/matplotlib.axes.Axes.plot.html)).

In [ ]:
# Create a figure object
fig = plt.figure()

# Add subplot (just 1) and set x and y limits based on data.
# 111 means "1x1 grid, 1st subplot".
# ax is the object containing objects to be plotted.
ax = fig.add_subplot(111, autoscale_on=False, xlim=(-0.1, max(x)*1.1), ylim=(-0.1,max(y)*1.1))
ax.grid()
ax.set_xlabel("x(t) [m]")
ax.set_ylabel("y(t) [m]")
plt.title(f"Trajectory of a projectile with $h=${h:.1f} m, $v_0=${v0:.1f} m/s, $\Theta_0=$ {theta:.1f}$^\circ$")

# Try: '--', '-', 'x', '.', 'o'
ax.plot(x, y, 'o', lw=2)

plt.show()

Now we use the [`FuncAnimation`](https://matplotlib.org/api/_as_gen/matplotlib.animation.FuncAnimation.html) to animate the plot. The process consists in 3 steps.
1. Plot the initial state of the plot. In our case we display:
  1. `ball` to represent the position of the projectile, which we initially place nowhere (`[], []`);
  1. `line` to represent the trajectory, which is initially pointlike (`x[0]` and `y[0]`);
  1. `info_text` to tell the user the simulation time and the coordinates of the projectile, which is initially an empty string (`''`).
1. Define a function to call at each frame; it takes an argument and is called to update all evolving objects (`ball`, `line`, `info_text`) appearing in the plot oject (`fig`).
1. Call the `FuncAnimation` function that updates the figure by calling the function of point 2 a number of times
  - `FuncAnimation` has  a number of useful options such as whether to repeat the animation, change the frame rate, introduce a delay between repetitions, etc.

## Now let's plot the projectile, the trajectory, and add written information reporting the time and the position of the projectile

In [ ]:
# Create a figure object
fig = plt.figure()

# Add subplot (just 1) and set x and y limits based on data
# ax is the object containing objects to be plotted
ax = fig.add_subplot(111, autoscale_on=False, xlim=(-0.1, max(x)*1.1), ylim=(-0.1,max(y)*1.1))
ax.grid()
ax.set_xlabel("x(t) [m]")
ax.set_ylabel("y(t) [m]")
plt.title(f"Trajectory of a projectile with $h=${h:.1f} m, $v_0=${v0:.1f} m/s, $\Theta_0=$ {theta:.1f}$^\circ$");

# If the inline plot has a scroll bar,
# click on the space on the left hand side
# of the plot to expand the whole plot

In [ ]:
# 1. Plot the initial state of the plot
#    - The projectile
ball, *_ = ax.plot([], [], 'o-', lw=2, color='red')
#    - The trajectory
line, *_ = ax.plot(x[0], y[0], '--', lw=1, color='orange')
#    - The information box
info_template = 'time = %.1fs  x = %.1fm   y = %.1fm'
info_text = ax.text(0.05, 0.95, '', transform=ax.transAxes)

We exploited the `_` variable to use just the first (and only) object contained in the list returned by `ax.plot`.
[See how the following block changes if you remove the `*_` in the block above.]  An equivalent syntax is, of course,
```Python
ball = ax.plot([], [], 'o-', lw=2, color='red')[0]
line = ax.plot(x[0], y[0], '--', lw=1, color='orange')[0]
```

In [ ]:
print(type(ball))
print(ball)

print(type(line))
print(line)

Now we can define the function to be called a number of times to update what needs to be shown.

The actual data is updated in the animate function `update_plots` as with the positions.

For the trajectory we plot all the point up to $i$-th position with `x[:i]` and `y[:i]` in `update_plots(i)`.

In [ ]:
# 2. Define an "update_plots" function to call at each frame.
# At each step we do 3 things
# 1) draw the line from 0 -> i-th position
# 2) draw the point at i-th position
# 3) update the text box with the time and position at i-th position

def update_plots(i):
    ball.set_data(x[i:i+1], y[i:i+1]) # set_data wants iterables as of Python 3.12, so x[i], y[i] will not work
    line.set_data(x[:i], y[:i])
    # Provide the numerical data to info_template to form an actual string
    info_text.set_text(info_template % (t[i], x[i], y[i]))
    # Return a tuple with the trajectory, projectile, and text
    return line, ball, info_text

In [ ]:
help(animation.FuncAnimation)

In [ ]:
# 3. Call the FuncAnimation function to redraw the 'fig' object using the 'update_plots' function with argument 
# which is an int given by np.arange(1, len(x))
anim = animation.FuncAnimation(fig, update_plots, arange(1, len(x)), interval=50, repeat=True)

# If only the first frame shows, try the next command or the block of code that follows
plt.show()

# Here is the rendering that makes it work on Colab
#from matplotlib import rc
# Try rc('animation', html='jshtml') if the following line is not succesful
#rc('animation', html='html5')
#anim

## Exercises you can practice with
- Write position *x* of the max height and put an arrow pointing to the apex
- Write the value of *x* and *y* on the plot near the actual position in real time
- Add a slider to modify the value of some parameters interactively
- Create an animated histogram for a Gaussian distribution
- Extend the problem to 3D and use 3D plot to show the trajectory in space using [mplot3d](https://matplotlib.org/mpl_toolkits/mplot3d/tutorial.html)

## Additional material
- Take a look at this very nice example of the animation of a [double pendulum](https://matplotlib.org/gallery/animation/double_pendulum_sgskip.html)

# READY FOR `examples/Python/5-NumPy.ipynb`!